# 📧 Lead Enrichment — Find Emails for Existing Leads
Upload your existing `multi_leads_*.xlsx` and this notebook finds emails
for every lead that doesn't have one yet — checking the business website,
contact/about pages, and the Yell listing page.

Great for enriching leads from before the v2 scraper upgrade.

1. Run Cell 1 — upload spreadsheet, finds emails, downloads updated file

In [ ]:
# =====================================================================
#  Lead Email Enricher — L&D Designs
#  Finds emails for existing leads that have no email address yet.
#  Upload any multi_leads_*.xlsx, downloads an enriched version.
# =====================================================================
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'requests', 'beautifulsoup4', 'openpyxl', 'lxml'])

import re, time
from urllib.parse import urljoin
from datetime import datetime
import requests
from bs4 import BeautifulSoup
import openpyxl
from openpyxl.styles import PatternFill
from google.colab import files

# -- Config ------------------------------------------------------------
BROWSER_HDR = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'}
SOCIAL_DOMAINS = ('facebook.com', 'instagram.com', 'twitter.com', 'tiktok.com', 'linkedin.com', 'linktree.com')
CONTACT_PATHS = [
    '/contact', '/contact-us', '/contact-us.html', '/contact.html',
    '/about', '/about-us', '/about.html', '/get-in-touch', '/reach-us',
]
EMAIL_RE = re.compile(r'\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b')
SKIP_EM = {'noreply', 'no-reply', 'example', 'test', 'wordpress', 'sentry',
           'privacy', 'abuse', 'postmaster', 'webmaster', 'do-not-reply', 'donotreply'}

def _get_html(url):
    try:
        r = requests.get(url, headers=BROWSER_HDR, timeout=10, allow_redirects=True)
        if r.status_code == 200: return r.text
    except: pass
    return ''

def _extract_email(html):
    if not html: return ''
    soup = BeautifulSoup(html, 'lxml')
    # Priority 1: explicit mailto links
    for tag in soup.find_all('a', href=re.compile(r'^mailto:', re.I)):
        m = re.search(r'mailto:([^?&\s]+)', tag['href'])
        if m:
            candidate = m.group(1).strip().lower()
            if not any(s in candidate for s in SKIP_EM) and '@' in candidate:
                return candidate
    # Priority 2: regex in page text
    for candidate in EMAIL_RE.findall(html):
        if not any(s in candidate.lower() for s in SKIP_EM) and '.' in candidate.split('@')[-1]:
            return candidate.lower()
    return ''

def find_email(website_url, yell_url=''):
    email = ''
    if website_url and not any(d in str(website_url).lower() for d in SOCIAL_DOMAINS):
        pages = [website_url] + [urljoin(website_url, p) for p in CONTACT_PATHS]
        for page_url in pages:
            if email: break
            email = _extract_email(_get_html(page_url))
    if not email and yell_url:
        email = _extract_email(_get_html(str(yell_url)))
    return email

# -- Upload ------------------------------------------------------------
print('Upload your multi_leads_*.xlsx file...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

wb = openpyxl.load_workbook(filename)
ws = wb.active
headers = [str(cell.value or '').strip() for cell in ws[1]]
print('Columns found:', headers)
print('Total rows:', ws.max_row - 1)

# Find column indices (0-based for lists, 1-based for openpyxl)
def col_idx(names):
    for n in names:
        for i, h in enumerate(headers):
            if n.lower() in h.lower():
                return i
    return -1

email_col    = col_idx(['Email'])
website_col  = col_idx(['Original Website', 'Website'])
yell_col     = col_idx(['Yell Listing', 'Yell'])
name_col     = col_idx(['Business Name', 'Name'])

print(f'\nColumn positions: Email={email_col}, Website={website_col}, Yell={yell_col}, Name={name_col}')

if email_col == -1:
    print('ERROR: No Email column found. Check your spreadsheet headers.')
else:
    # -- Enrich --------------------------------------------------------
    no_email_rows = []
    for row_idx in range(2, ws.max_row + 1):
        email_val = ws.cell(row_idx, email_col + 1).value
        if not email_val or str(email_val).strip() in ('', 'None', 'nan'):
            no_email_rows.append(row_idx)

    print(f'Leads with no email: {len(no_email_rows)} / {ws.max_row - 1}')
    print('Starting email search...\n')

    found = 0
    HIGHLIGHT = PatternFill('solid', fgColor='C8E6C9')

    for count, row_idx in enumerate(no_email_rows, 1):
        name    = ws.cell(row_idx, name_col + 1).value if name_col >= 0 else '?'
        website = ws.cell(row_idx, website_col + 1).value if website_col >= 0 else ''
        yell    = ws.cell(row_idx, yell_col + 1).value if yell_col >= 0 else ''

        email = find_email(str(website or ''), str(yell or ''))

        if email:
            ws.cell(row_idx, email_col + 1).value = email
            ws.cell(row_idx, email_col + 1).fill = HIGHLIGHT
            found += 1
            print(f'  [{count}/{len(no_email_rows)}] FOUND: {name} => {email}')
        else:
            if count % 50 == 0:
                print(f'  [{count}/{len(no_email_rows)}] checked...')

        # Polite delay every 10 requests
        if count % 10 == 0:
            time.sleep(1)

    print(f'\nDone! Found {found} new emails out of {len(no_email_rows)} leads checked.')
    print(f'Email coverage: {round(found / len(no_email_rows) * 100, 1)}% of previously missing leads\n')

    # -- Save ----------------------------------------------------------
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_fname = 'enriched_leads_' + ts + '.xlsx'
    wb.save(out_fname)
    files.download(out_fname)
    print(f'Saved and downloading: {out_fname}')
    print('Green highlighting = newly found emails')
